# 01 - Build the Pipeline We Will Profile

This notebook builds the synthetic deep learning workflow that the workshop will profile with Nsight Systems. The workload is intentionally ordinary-looking: a CPU input pipeline, host-to-device transfer, an MLP classifier, micro-batched training, and metric logging.

The important detail is that the pipeline contains four trace-visible patterns that often appear in real code:

1. Extra synchronization.
2. Short-lived kernels from small training chunks.
3. CPU/GPU handoffs that serialize work.
4. Frequent host/device movement from small batches and blocking transfers.

Notebook 02 moves these pieces into paired modules so attendees can fix one pattern at a time.


## Setup

The notebook chooses CUDA when it is available. On an L40S or A100, keep the CUDA-sized configuration. On CPU or a small laptop GPU, the configuration below automatically scales down so the notebook remains usable for exploration.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import subprocess
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if not (ROOT / "profiling_workshop").exists() and (ROOT.parent / "profiling_workshop").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from profiling_workshop.common import cuda_is_available_quietly, make_model, maybe_sync, nvtx_range, seed_everything
from profiling_workshop.pipeline.shared import RawSyntheticDataset, augment_features, labels_from_features

DEVICE = torch.device("cuda" if cuda_is_available_quietly() else "cpu")
print(f"Workshop root: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"Selected device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Configuration

The CUDA defaults are designed to create real matrix multiply work while still leaving room for the profiling pathologies to stand out. The CPU defaults are only a smoke-sized version of the same shape.


In [ ]:
@dataclass(frozen=True)
class NotebookConfig:
    seed: int = 123
    samples: int = 4096 if DEVICE.type == "cuda" else 256
    batch_size: int = 128 if DEVICE.type == "cuda" else 32
    features: int = 2048 if DEVICE.type == "cuda" else 64
    classes: int = 64 if DEVICE.type == "cuda" else 8
    hidden: int = 4096 if DEVICE.type == "cuda" else 128
    depth: int = 4 if DEVICE.type == "cuda" else 2
    micro_batches: int = 16 if DEVICE.type == "cuda" else 4
    cpu_work: int = 2
    lr: float = 1e-3


cfg = NotebookConfig()
seed_everything(cfg.seed)
cfg


## Raw Samples and CPU Feature Work

The synthetic data is deterministic. The raw samples are cheap enough to generate on demand, then a CPU feature-engineering pass makes the input pipeline visible in the timeline.


In [ ]:
raw_dataset = RawSyntheticDataset(samples=cfg.samples, features=cfg.features, seed=cfg.seed)
raw_batch = torch.stack([raw_dataset[i] for i in range(min(4, cfg.batch_size))])

x_preview = augment_features(raw_batch, cfg.cpu_work)
y_preview = labels_from_features(x_preview, cfg.classes)

print(x_preview.shape, y_preview.tolist())
print(f"preview mean={x_preview.mean().item():.4f} std={x_preview.std().item():.4f}")


## DataLoader

For the first pass, the loader is deliberately simple: no worker processes and no pinned host memory. This keeps the code readable before we split it into lab modules.


In [ ]:
loader = DataLoader(
    raw_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=True,
)

raw_batch = next(iter(loader))
print(raw_batch.shape)


## Model

The feature MLP supplies enough dense linear algebra to exercise a data-center GPU. The broadcast-distance head is plausible, but it creates less favorable kernel and memory behavior than the matmul formulation used in the reference solution.


In [ ]:
model = make_model(
    features=cfg.features,
    hidden=cfg.hidden,
    classes=cfg.classes,
    depth=cfg.depth,
    head="broadcast-distance",
    nvtx_enabled=True,
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"parameters={parameter_count:,}")


## One Training Batch

This cell is a compact version of the original pipeline. The NVTX range names map directly to the four lab sections in Notebook 02.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
loss_total = 0.0
correct_total = 0
seen = 0

with nvtx_range("issue_3_cpu_gpu_handoff_main_thread_preprocessing"):
    x_cpu = augment_features(raw_batch, cfg.cpu_work)
    y_cpu = labels_from_features(x_cpu, cfg.classes)

with nvtx_range("issue_4_batch_io_blocking_h2d"):
    x = x_cpu.to(DEVICE)
    y = y_cpu.to(DEVICE)

with nvtx_range("issue_1_extra_cuda_synchronize_after_h2d"):
    maybe_sync(DEVICE)

micro_batches = max(1, min(cfg.micro_batches, x.shape[0]))
for mb_x, mb_y in zip(torch.chunk(x, micro_batches), torch.chunk(y, micro_batches)):
    with nvtx_range("issue_2_short_lived_microbatch_train_step"):
        optimizer.zero_grad(set_to_none=False)
        logits = model(mb_x)
        loss = criterion(logits, mb_y)
        loss.backward()
        optimizer.step()

    with nvtx_range("issue_1_synchronizing_metrics_and_d2h"):
        maybe_sync(DEVICE)
        loss_total += float(loss.detach().item()) * mb_y.numel()
        pred_cpu = logits.detach().argmax(dim=1).cpu()
        correct_total += int((pred_cpu == mb_y.cpu()).sum().item())
        seen += mb_y.numel()

maybe_sync(DEVICE)
print(f"loss={loss_total / max(seen, 1):.4f} accuracy={correct_total / max(seen, 1):.3f}")


## Script Layout Used by the Lab

The lab version keeps the orchestrator small and moves each pattern into one focused module. The `problem` and `solution` packages have matching filenames and matching public function names.


In [ ]:
module_paths = [
    ROOT / "profiling_workshop" / "pipeline" / "problem" / "synchronization.py",
    ROOT / "profiling_workshop" / "pipeline" / "problem" / "short_kernels.py",
    ROOT / "profiling_workshop" / "pipeline" / "problem" / "handoffs.py",
    ROOT / "profiling_workshop" / "pipeline" / "problem" / "batching.py",
    ROOT / "profiling_workshop" / "pipeline" / "problem" / "orchestrator.py",
]
for path in module_paths:
    print(path.relative_to(ROOT))


## Run the Original Pipeline Script

This is the command Notebook 02 profiles. It prints an overall `RESULT` line plus one `REGION` line for each issue area. The region timers are only a quick wall-clock hint; Nsight Systems and the NVTX ranges are the source of truth.


In [ ]:
script = ROOT / "scripts" / "run_problem_pipeline.py"
smoke_args = [
    sys.executable,
    str(script),
    "--device", str(DEVICE),
    "--samples", "512" if DEVICE.type == "cuda" else "128",
    "--batch-size", "64" if DEVICE.type == "cuda" else "32",
    "--features", "512" if DEVICE.type == "cuda" else "64",
    "--hidden", "1024" if DEVICE.type == "cuda" else "128",
    "--depth", "2",
    "--classes", "16" if DEVICE.type == "cuda" else "8",
    "--micro-batches", "8" if DEVICE.type == "cuda" else "4",
    "--num-workers", "0",
    "--log-every", "0",
]
print("$ " + " ".join(smoke_args))
subprocess.run(smoke_args, check=True, text=True)


## Handoff

In Notebook 02, each section points to one of these modules, captures Nsight Systems traces for the editable problem pipeline and the reference solution pipeline, and compares the relevant NVTX ranges.
